# FIT5196 Assessment 1 - Group029 EDA

This notebook loads the six submitted standardised CSV files and recreates all assessed figures, reported statistics, findings and future machine-learning questions. It is designed to run offline from a fresh kernel.

## 0. Configuration and data loading

The configuration is relative to the project root. The literal string `NaN` remains visible during CSV loading so that prescribed missing-string values are not silently converted.

In [ ]:
from pathlib import Path

GROUP_ID = "Group029"
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError("Run this notebook from the Group029 project root or its notebooks folder.")

print({"group_id": GROUP_ID, "project_root": ".", "output_dir": "outputs", "figure_dir": "figures"})

In [ ]:
import platform
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.patches import Patch

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)

def read_table(name, id_columns):
    path = OUTPUT_DIR / f"{GROUP_ID}_{name}_standardised.csv"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, keep_default_na=False, dtype={column: "string" for column in id_columns})

customers = read_table("customers", ["customer_id"])
products = read_table("products", ["product_id"])
orders = read_table("orders", ["order_id", "source_system_record_id", "customer_id", "coupon_code", "promo_code"])
order_items = read_table("order_items", ["order_item_id", "order_id", "product_id"])
deliveries = read_table("deliveries", ["delivery_id", "order_id"])
product_reviews = read_table("product_reviews", ["review_id", "order_id", "order_item_id", "product_id", "customer_id"])

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"], format="%Y-%m-%d %H:%M:%S", errors="raise")

def to_bool(series, name):
    mapping = {"true": True, "false": False}
    converted = series.astype(str).str.lower().map(mapping)
    if converted.isna().any():
        raise ValueError(f"Unexpected boolean value in {name}")
    return converted.astype(bool)

deliveries["on_time_in_full"] = to_bool(deliveries["on_time_in_full"], "on_time_in_full")
product_reviews["contains_non_latin_script"] = to_bool(product_reviews["contains_non_latin_script"], "contains_non_latin_script")

tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "deliveries": deliveries,
    "product_reviews": product_reviews,
}

display(pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in tables.items()
]))
print({"python": platform.python_version(), "pandas": pd.__version__})

## 1. Context and data-preparation assurance

The allocated JSON and XML exports represent the same retail domain with different nesting, names and text conventions. The solution workflow first parses each source structurally, then standardises comparable fields before reconciling records by stable business keys. Four material decisions are especially important:

- **Relational grain is preserved.** Customers, products, orders, order items, deliveries and reviews remain separate one-to-many tables rather than being forced into one duplicated flat table.
- **Normalisation precedes reconciliation.** IDs retain case and leading zeros; dates, timestamps, booleans, currency and percentages follow the public contract before cross-source comparison.
- **Published arithmetic is reproduced.** Rounded line revenue is aggregated to order price; included GST is reported separately; coupon discount and delivery charge are applied in the required order.
- **Narrative cleaning is bounded.** JSON/XML structure is parsed before regex is used for tags, markers, URLs, wrappers and references. Multilingual clean text is preserved, with a separate Latin-analysis field.

The full 111-row lineage is recorded in `Group029_source_to_target_mapping.csv`. Material executable checks in the solution notebook include `VAL-FLOW-ORD-01`, `VAL-FLOW-ORD-02`, `VAL-ARITH-TOTAL-01`, `VAL-PK-REV-01`, `VAL-FK-REV-01` and `VAL-TEXT-REV-01`.

In [ ]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "deliveries": "delivery_id",
    "product_reviews": "review_id",
}

assurance_rows = []
for table_name, key in primary_keys.items():
    frame = tables[table_name]
    semantic_missing = frame[key].astype(str).str.strip().str.lower().isin({"", "nan", "null", "none", "<na>"}).sum()
    assurance_rows.append({
        "check": f"{table_name}.{key}",
        "observed_rows": len(frame),
        "missing_primary_keys": int(semantic_missing),
        "duplicate_primary_keys": int(frame[key].duplicated().sum()),
        "status": "PASS" if semantic_missing == 0 and not frame[key].duplicated().any() else "FAIL",
    })

foreign_keys = [
    ("orders.customer_id", orders["customer_id"], customers["customer_id"]),
    ("order_items.order_id", order_items["order_id"], orders["order_id"]),
    ("order_items.product_id", order_items["product_id"], products["product_id"]),
    ("deliveries.order_id", deliveries["order_id"], orders["order_id"]),
    ("product_reviews.order_id", product_reviews["order_id"], orders["order_id"]),
    ("product_reviews.order_item_id", product_reviews["order_item_id"], order_items["order_item_id"]),
    ("product_reviews.product_id", product_reviews["product_id"], products["product_id"]),
    ("product_reviews.customer_id", product_reviews["customer_id"], customers["customer_id"]),
]
fk_rows = []
for label, child, parent in foreign_keys:
    orphan_count = int((~child.isin(set(parent))).sum())
    fk_rows.append({"relationship": label, "orphan_rows": orphan_count, "status": "PASS" if orphan_count == 0 else "FAIL"})

display(pd.DataFrame(assurance_rows))
display(pd.DataFrame(fk_rows))
if any(row["status"] == "FAIL" for row in assurance_rows + fk_rows):
    raise ValueError("EDA input assurance failed.")

## 2. Assessed EDA visualisations

Only Figures 1-8 below are assessed. Together they cover all six required analytical categories, all six output tables and two explicitly checked relational analyses.

### Figure 1: Customer composition by loyalty tier

**Question:** What is the composition of the customer base by loyalty tier?  
**Observation unit and denominator:** One canonical customer; all 500 customers.  
**Tables and join keys:** `customers`; no join.  
**Interpretation and limitation:** The chart describes the current customer mix. Tier size does not itself indicate profitability or engagement, and the tiers may reflect rules not included in the data.

In [ ]:
tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
figure1_summary = (
    customers.groupby("loyalty_tier", as_index=False)
    .agg(customers=("customer_id", "size"), median_prior_orders=("prior_12m_orders", "median"), median_prior_value=("lifetime_value_before_period", "median"))
    .set_index("loyalty_tier")
    .reindex(tier_order)
    .reset_index()
)
figure1_summary["customer_pct"] = figure1_summary["customers"] / len(customers) * 100
display(figure1_summary)

fig, ax = plt.subplots(figsize=(8.5, 5.2))
bars = ax.bar(figure1_summary["loyalty_tier"], figure1_summary["customer_pct"], color=["#9A6A3A", "#778899", "#C49A00", "#4F46E5"])
ax.set_title("Figure 1. Customer composition by loyalty tier", loc="left", weight="bold")
ax.set_xlabel("Loyalty tier")
ax.set_ylabel("Customers (%)")
ax.set_ylim(0, max(figure1_summary["customer_pct"]) * 1.22)
ax.bar_label(bars, labels=[f"{row.customer_pct:.1f}%\n(n={row.customers})" for row in figure1_summary.itertuples()], padding=3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure1_customer_loyalty_composition.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 2: Product price distribution by category

**Question:** How does the listed product-price distribution vary by category?  
**Observation unit and denominator:** One canonical product; all 1,000 products, with 100 products in each category.  
**Tables and join keys:** `products`; no join.  
**Interpretation and limitation:** The comparison describes catalogue positioning, not realised selling prices or demand. Outliers are hidden in the figure for readability but retained in the summary statistics.

In [ ]:
figure2_summary = (
    products.groupby("category", as_index=False)
    .agg(products=("product_id", "size"), median_price=("unit_price", "median"), q1_price=("unit_price", lambda x: x.quantile(0.25)), q3_price=("unit_price", lambda x: x.quantile(0.75)))
    .sort_values("median_price", ascending=False)
)
display(figure2_summary)

category_order = figure2_summary["category"].tolist()
price_groups = [products.loc[products["category"].eq(category), "unit_price"] for category in category_order]
fig, ax = plt.subplots(figsize=(11, 6))
box = ax.boxplot(price_groups, tick_labels=category_order, showfliers=False, patch_artist=True, medianprops={"color": "#7F1D1D", "linewidth": 2})
for patch in box["boxes"]:
    patch.set_facecolor("#93C5FD")
ax.set_title("Figure 2. Product price distribution by category", loc="left", weight="bold")
ax.set_xlabel("Product category")
ax.set_ylabel("Unit price (AUD)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure2_product_price_by_category.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 3: Order value by sales channel and coupon use

**Question:** How does order value vary by sales channel and coupon use?  
**Observation unit and denominator:** One canonical order; all 5,000 orders, with group sizes reported below.  
**Tables and join keys:** `orders`; no join.  
**Interpretation and limitation:** Medians and IQRs are compared without claiming causation. `order_total` is post-discount, coupon allocation is not random, and outliers are hidden only in the chart.

In [ ]:
orders["coupon_used"] = orders["coupon_discount"].gt(0).map({True: "Coupon used", False: "No coupon"})
channel_order = sorted(orders["sales_channel"].unique())
coupon_order = ["No coupon", "Coupon used"]
figure3_summary = orders.groupby(["sales_channel", "coupon_used"], as_index=False).agg(orders=("order_id", "size"), median_order_total=("order_total", "median"), mean_order_total=("order_total", "mean"))
display(figure3_summary)

fig, ax = plt.subplots(figsize=(10, 6))
positions, values, colours = [], [], []
base_positions = list(range(1, len(channel_order) + 1))
offsets = {"No coupon": -0.18, "Coupon used": 0.18}
palette = {"No coupon": "#64748B", "Coupon used": "#0F766E"}
for channel_index, channel in enumerate(channel_order, start=1):
    for coupon_label in coupon_order:
        positions.append(channel_index + offsets[coupon_label])
        values.append(orders.loc[orders["sales_channel"].eq(channel) & orders["coupon_used"].eq(coupon_label), "order_total"])
        colours.append(palette[coupon_label])
box = ax.boxplot(values, positions=positions, widths=0.30, showfliers=False, patch_artist=True, medianprops={"color": "white", "linewidth": 1.8})
for patch, colour in zip(box["boxes"], colours):
    patch.set_facecolor(colour)
ax.set_xticks(base_positions, channel_order)
ax.set_title("Figure 3. Order value by sales channel and coupon use", loc="left", weight="bold")
ax.set_xlabel("Sales channel")
ax.set_ylabel("Order total (AUD)")
ax.legend(handles=[Patch(facecolor=palette[label], label=label) for label in coupon_order], frameon=False)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure3_order_value_by_channel_and_coupon.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 4: Monthly order activity and revenue

**Question:** How do monthly order counts and revenue vary across 2018?  
**Observation unit and denominator:** Canonical orders aggregated to calendar month; all 5,000 orders.  
**Tables and join keys:** `orders`; no join.  
**Interpretation and limitation:** Order-count peaks need not coincide with revenue peaks because order value varies. Only one historical year is available, month lengths differ, and the pattern is not evidence of stable seasonality.

In [ ]:
monthly = (
    orders.assign(month=orders["order_timestamp"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(order_count=("order_id", "size"), revenue=("order_total", "sum"), mean_order_total=("order_total", "mean"))
)
display(monthly)

fig, axes = plt.subplots(2, 1, figsize=(10.5, 7.5), sharex=True)
axes[0].plot(monthly["month"], monthly["order_count"], marker="o", linewidth=2.2, color="#0F766E")
axes[1].plot(monthly["month"], monthly["revenue"], marker="o", linewidth=2.2, color="#1D4ED8")
fig.suptitle("Figure 4. Monthly order activity and revenue", x=0.08, y=0.99, ha="left", weight="bold")
axes[0].set_ylabel("Orders")
axes[1].set_ylabel("Revenue (AUD)")
axes[1].set_xlabel("Month")
axes[1].tick_params(axis="x", rotation=45)
axes[1].yaxis.set_major_formatter(lambda value, _: f"${value/1_000_000:.2f}m")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIGURE_DIR / "Figure4_monthly_order_activity_and_revenue.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 5: Review rating composition and length

**Question:** How are review ratings distributed, and does review length differ materially by rating?  
**Observation unit and denominator:** One canonical product review; all 7,000 reviews.  
**Tables and join keys:** `product_reviews`; no join.  
**Interpretation and limitation:** Rating composition and word counts describe submitted reviews only. They do not represent purchasers who left no review, and review length is not a direct measure of review quality.

In [ ]:
rating_order = [1, 2, 3, 4, 5]
figure5_summary = (
    product_reviews.groupby("rating", as_index=False)
    .agg(reviews=("review_id", "size"), median_words=("review_word_count", "median"), mean_helpful_votes=("helpful_votes", "mean"), non_latin_share=("contains_non_latin_script", "mean"))
    .set_index("rating")
    .reindex(rating_order)
    .reset_index()
)
figure5_summary["review_pct"] = figure5_summary["reviews"] / len(product_reviews) * 100
figure5_summary["non_latin_pct"] = figure5_summary["non_latin_share"] * 100
display(figure5_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
bars = axes[0].bar(figure5_summary["rating"].astype(str), figure5_summary["review_pct"], color="#2563EB")
axes[0].bar_label(bars, labels=[f"{value:.1f}%" for value in figure5_summary["review_pct"]], padding=3)
axes[0].set_xlabel("Rating (1-5)")
axes[0].set_ylabel("Reviews (%)")
axes[0].set_ylim(0, figure5_summary["review_pct"].max() * 1.20)

word_groups = [product_reviews.loc[product_reviews["rating"].eq(rating), "review_word_count"] for rating in rating_order]
word_box = axes[1].boxplot(word_groups, tick_labels=[str(rating) for rating in rating_order], showfliers=False, patch_artist=True, medianprops={"color": "#7F1D1D", "linewidth": 2})
for patch in word_box["boxes"]:
    patch.set_facecolor("#A7F3D0")
axes[1].set_xlabel("Rating (1-5)")
axes[1].set_ylabel("Review word count")
fig.suptitle("Figure 5. Review rating composition and length", x=0.06, y=1.02, ha="left", weight="bold")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure5_review_rating_and_length.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 6: Product-category demand and review outcomes

**Question:** Which product categories combine stronger unit demand with stronger or weaker review outcomes?  
**Observation units and denominators:** Order items are aggregated to category for 15,706 canonical order-item rows; reviews are aggregated separately for 7,000 canonical reviews.  
**Tables and join keys:** `order_items.product_id` → `products.product_id` and `product_reviews.product_id` → `products.product_id`.  
**Join check:** Each fact table is joined separately to the one-row-per-product dimension before category aggregation, preventing an order-item × review many-to-many multiplication.  
**Interpretation and limitation:** Units sold and ratings are descriptive. Product availability, exposure, price and review-selection effects are not controlled.

In [ ]:
item_product = order_items.merge(products[["product_id", "category"]], on="product_id", how="left", validate="many_to_one")
review_product = product_reviews.merge(products[["product_id", "category"]], on="product_id", how="left", validate="many_to_one")

figure6_join_check = pd.DataFrame([
    {"join": "order_items -> products", "rows_before": len(order_items), "rows_after": len(item_product), "unique_entities_before": order_items["order_item_id"].nunique(), "unique_entities_after": item_product["order_item_id"].nunique()},
    {"join": "product_reviews -> products", "rows_before": len(product_reviews), "rows_after": len(review_product), "unique_entities_before": product_reviews["review_id"].nunique(), "unique_entities_after": review_product["review_id"].nunique()},
])
display(figure6_join_check)
if not ((figure6_join_check["rows_before"] == figure6_join_check["rows_after"]).all() and (figure6_join_check["unique_entities_before"] == figure6_join_check["unique_entities_after"]).all()):
    raise ValueError("Figure 6 join changed a fact-table grain.")

review_product["low_rating"] = review_product["rating"].le(2)
category_sales = item_product.groupby("category", as_index=False).agg(order_lines=("order_item_id", "size"), units_sold=("quantity", "sum"), revenue=("line_revenue", "sum"))
category_reviews = review_product.groupby("category", as_index=False).agg(reviews=("review_id", "size"), mean_rating=("rating", "mean"), low_rating_rate=("low_rating", "mean"))
figure6_summary = category_sales.merge(category_reviews, on="category", validate="one_to_one").sort_values("units_sold", ascending=True)
figure6_summary["low_rating_pct"] = figure6_summary["low_rating_rate"] * 100
display(figure6_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 6.5), sharey=True)
axes[0].barh(figure6_summary["category"], figure6_summary["units_sold"], color="#0F766E")
axes[0].set_xlabel("Units sold")
axes[0].set_ylabel("Product category")
axes[0].set_xlim(0, figure6_summary["units_sold"].max() * 1.12)
for y, value in enumerate(figure6_summary["units_sold"]):
    axes[0].text(value + 35, y, f"{value:,}", va="center", fontsize=8)

axes[1].barh(figure6_summary["category"], figure6_summary["mean_rating"], color="#F59E0B")
axes[1].set_xlabel("Mean review rating (1-5)")
axes[1].set_xlim(0, 5)
for y, value in enumerate(figure6_summary["mean_rating"]):
    axes[1].text(value + 0.05, y, f"{value:.2f}", va="center", fontsize=8)
fig.suptitle("Figure 6. Product-category demand and review outcomes", x=0.06, y=0.99, ha="left", weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.96))
fig.savefig(FIGURE_DIR / "Figure6_category_demand_and_reviews.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 7: On-time-in-full performance by carrier and service level

**Question:** How does `on_time_in_full` performance vary across carriers and service levels?  
**Observation unit and denominator:** One canonical delivery; all 5,000 deliveries, with subgroup counts reported below.  
**Tables and join keys:** `deliveries`; no join.  
**Interpretation and limitation:** Carrier/service groups can differ in route mix, distance and destination. The comparison is descriptive rather than a causal carrier ranking.

In [ ]:
deliveries_eda = deliveries.copy()
deliveries_eda["delayed"] = deliveries_eda["delay_days"].gt(0)
figure7_summary = deliveries_eda.groupby(["carrier", "service_level"], as_index=False).agg(deliveries=("delivery_id", "size"), otif_rate=("on_time_in_full", "mean"), delayed_rate=("delayed", "mean"), mean_delay_days=("delay_days", "mean"))
figure7_summary["otif_pct"] = figure7_summary["otif_rate"] * 100
figure7_summary["delayed_pct"] = figure7_summary["delayed_rate"] * 100
display(figure7_summary.sort_values(["service_level", "carrier"]))

carrier_order = sorted(figure7_summary["carrier"].unique())
service_order = ["Standard", "Express"]
rate_matrix = figure7_summary.pivot(index="carrier", columns="service_level", values="otif_pct").reindex(index=carrier_order, columns=service_order)
count_matrix = figure7_summary.pivot(index="carrier", columns="service_level", values="deliveries").reindex(index=carrier_order, columns=service_order)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
image = ax.imshow(rate_matrix.values, cmap="YlGnBu", vmin=80, vmax=100, aspect="auto")
for row in range(rate_matrix.shape[0]):
    for column in range(rate_matrix.shape[1]):
        ax.text(column, row, f"{rate_matrix.iloc[row, column]:.1f}%\n(n={int(count_matrix.iloc[row, column])})", ha="center", va="center", color="black")
ax.set_xticks(range(len(service_order)), service_order)
ax.set_yticks(range(len(carrier_order)), carrier_order)
ax.set_xlabel("Service level")
ax.set_ylabel("Carrier")
ax.set_title("Figure 7. On-time-in-full rate by carrier and service level", loc="left", weight="bold")
colour_bar = fig.colorbar(image, ax=ax)
colour_bar.set_label("OTIF rate (%)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure7_otif_by_carrier_and_service_level.png", dpi=200, bbox_inches="tight")
plt.show()

print({"delivery_count": len(deliveries_eda), "delayed_deliveries": int(deliveries_eda["delayed"].sum()), "overall_otif_pct": round(deliveries_eda["on_time_in_full"].mean() * 100, 1)})

### Figure 8: Delivery delay rate by customer loyalty tier

**Question:** Does delivery delay rate differ across customer loyalty tiers?  
**Observation unit and denominator:** One canonical order/delivery; all 5,000 matched orders and deliveries.  
**Tables and join keys:** `orders.customer_id` → `customers.customer_id`, then `orders.order_id` → `deliveries.order_id`.  
**Join check:** The joined row count and unique `order_id` count must remain equal to the original order count.  
**Interpretation and limitation:** Loyalty tier is not a causal driver of delivery performance. Location, product mix and service selection may confound differences.

In [ ]:
delivery_customer = (
    orders[["order_id", "customer_id"]]
    .merge(customers[["customer_id", "loyalty_tier"]], on="customer_id", how="left", validate="many_to_one")
    .merge(deliveries_eda[["order_id", "delay_days", "delayed"]], on="order_id", how="inner", validate="one_to_one")
)

figure8_join_check = pd.DataFrame([{
    "orders_before_join": len(orders),
    "rows_after_join": len(delivery_customer),
    "unique_orders_after_join": delivery_customer["order_id"].nunique(),
}])
display(figure8_join_check)
if not (len(delivery_customer) == len(orders) == delivery_customer["order_id"].nunique()):
    raise ValueError("Figure 8 join changed the one-row-per-order grain.")

figure8_summary = delivery_customer.groupby("loyalty_tier", as_index=False).agg(orders=("order_id", "size"), delayed_orders=("delayed", "sum"), delay_rate=("delayed", "mean"), mean_delay_days=("delay_days", "mean"))
figure8_summary["delay_pct"] = figure8_summary["delay_rate"] * 100
figure8_summary["loyalty_tier"] = pd.Categorical(figure8_summary["loyalty_tier"], categories=tier_order, ordered=True)
figure8_summary = figure8_summary.sort_values("loyalty_tier")
display(figure8_summary)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
bars = ax.bar(figure8_summary["loyalty_tier"].astype(str), figure8_summary["delay_pct"], color="#DC2626")
ax.set_title("Figure 8. Delivery delay rate by customer loyalty tier", loc="left", weight="bold")
ax.set_xlabel("Customer loyalty tier")
ax.set_ylabel("Delayed deliveries (%)")
ax.set_ylim(0, figure8_summary["delay_pct"].max() * 1.25)
ax.bar_label(bars, labels=[f"{row.delay_pct:.1f}%\n(n={row.orders})" for row in figure8_summary.itertuples()], padding=3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure8_delay_rate_by_loyalty_tier.png", dpi=200, bbox_inches="tight")
plt.show()

### Assessed-figure coverage register

| Figure | Required category coverage | Output tables used | Relational analysis |
|---|---|---|---|
| 1 | Univariate composition | customers | No |
| 2 | Bivariate group comparison | products | No |
| 3 | Multivariate/segmented relationship | orders | No |
| 4 | Temporal pattern | orders | No |
| 5 | Review/text behaviour; bivariate relationship | product_reviews | No |
| 6 | Multivariate/segmented relationship; review behaviour | order_items, products, product_reviews | Yes - two fact tables aggregated separately |
| 7 | Delivery/operational performance; bivariate comparison | deliveries | No |
| 8 | Delivery/operational performance; multivariate relationship | orders, customers, deliveries | Yes - checked sequential joins |

The eight figures collectively cover all six required categories, all six output tables, and two checked relational analyses.

## 3. Ten evidence-based findings

1. **Customer tiers are concentrated below Platinum (Figure 1).** At the customer grain, Bronze is the largest tier with 191 of 500 customers (38.2%), followed by Silver with 163 (32.6%); Platinum contains 43 (8.6%). This matters for targeting and capacity estimates, but tier definitions are not supplied and tier size does not measure value. The retailer should compare tier rules and outcomes before using tier membership operationally.

2. **Catalogue prices differ strongly by category (Figure 2).** Each category contains 100 products, yet the median listed price ranges from AUD 227.97 for Accessory to AUD 2,585.08 for Home Entertainment. The balanced product counts make the category comparison transparent, but listed prices are not transaction-weighted realised prices. Stock availability, discounting and product mix should be incorporated before making revenue forecasts.

3. **Coupon orders have lower post-discount medians in every channel (Figure 3).** Across all 5,000 orders, coupon groups contain 1,020 Mobile, 1,103 Store and 1,102 Web orders. Their median totals are about AUD 411-519 below the corresponding no-coupon medians. This is not evidence that coupons reduce purchasing because the outcome mechanically includes the discount and coupon allocation is non-random. Incrementality needs a controlled or carefully matched design.

4. **The highest-volume month is not the highest-revenue month (Figure 4).** September has the most orders (457), while December has the highest revenue (AUD 1,447,431.47 from 454 orders). This shows that monthly revenue depends on order value and mix as well as volume. One historical year and unequal month lengths prevent a stable seasonality claim; item/category composition should be examined before inventory or staffing changes.

5. **Ratings are concentrated at the high end, while review length is broadly stable (Figure 5).** Ratings 4-5 account for 4,532 of 7,000 reviews (64.7%), whereas ratings 1-2 account for 1,263 (18.0%). Median review length remains between 148 and 155 words across all ratings, so length alone shows little separation. Purchasers who did not review are absent, and rating distributions may reflect selection or solicitation effects.

6. **Average rating varies by product category (Figure 6).** Across 7,000 reviews, category means range from 3.26 for Laptop (678 reviews) to 4.21 for Smartphone (694 reviews). Similar review counts reduce simple sample-size concerns, but brand, price, product age and purchaser mix may explain the differences. Product teams should inspect within-category and product-level variation before treating category as the cause.

7. **Sales volume and average rating do not form a simple trade-off (Figure 6).** Accessory has the highest unit volume (2,899 units) with a mean rating of 4.10, while Home Entertainment has 1,320 units and a similar mean rating of 4.09. The analysis separately aggregates 15,706 order items and 7,000 reviews before joining category summaries, avoiding fact-to-fact multiplication. Exposure and availability remain unobserved, so this is descriptive rather than demand causation.

8. **Most deliveries are on time in full, but delays remain operationally material (Figure 7).** Among 5,000 canonical deliveries, 4,460 (89.2%) are on time in full and 540 (10.8%) are delayed. This supports routine exception monitoring, but the data do not isolate controllable carrier causes from route, distance or order characteristics.

9. **Express OTIF differs across carriers (Figure 7).** DHL Express records 92.8% OTIF across 223 deliveries, compared with 84.9% for StarTrack Express across 218, a gap of about 7.9 percentage points. Standard-service rates are much closer (about 89.0%-90.1%). Route and destination mix could explain part of the gap, so carrier allocation should not change without adjusted analysis or a monitored trial.

10. **Observed delay rates differ by loyalty tier (Figure 8).** At one order/delivery per row, Bronze records 177 delays among 1,923 orders (9.2%), versus 59 among 456 Platinum orders (12.9%); the checked joins preserve all 5,000 orders. Loyalty tier should not be interpreted as causal or used to deprioritise customers. Operational variables should be tested first, with fairness monitoring across tiers.

## 4. Five future machine-learning questions

### ML Question 1: Can weekly order volume by sales channel be forecast for fulfilment planning?

| Element | Response |
|---|---|
| EDA evidence | Figure 4 shows material month-to-month volume variation, while Figure 3 confirms three active channels. |
| Decision and unit | Allocate short-term warehouse labour and channel support; one week × sales channel. |
| Type and target | Forecasting; next-week order count for each channel. |
| Decision-time predictors | Lagged weekly counts, rolling statistics, calendar week/month/season and channel, using only data complete before the forecast cutoff. |
| Validation and metric | Expanding-window temporal validation; MAE and WAPE against a seasonal or last-period baseline. |
| Risks | Future orders must not enter rolling features. One year limits seasonal learning, and holidays, campaigns, stock constraints and later drift are not observed. |

### ML Question 2: Can low-rating review risk be identified after fulfilment but before a review is submitted?

| Element | Response |
|---|---|
| EDA evidence | Figures 5-6 show an 18.0% low-rating share and category-level rating differences. |
| Decision and unit | Prioritise post-purchase service; one fulfilled order item that may receive a review. |
| Type and target | Binary classification; whether the eventual rating is 1-2. |
| Decision-time predictors | Product category/price, order value, delivery performance and customer history known by the post-fulfilment decision time. |
| Validation and metric | Time-based split by fulfilment/review period; PR-AUC and recall for low ratings, with calibration checks. |
| Risks | Review text, rating-derived fields and post-review helpful votes are leakage. Non-reviewers create selection bias, and customer-group performance should be audited for fairness. |

### ML Question 3: Can future product demand be estimated for assortment and replenishment decisions?

| Element | Response |
|---|---|
| EDA evidence | Figures 2 and 6 show large category differences in price and observed unit volume. |
| Decision and unit | Set replenishment priorities; one product × future planning period. |
| Type and target | Regression/forecasting; units sold in the next period. |
| Decision-time predictors | Product category, subcategory, brand, price, cost, launch age, warranty, prior-period unit demand and season, all frozen before the prediction cutoff. |
| Validation and metric | Rolling temporal validation by planning period; MAE/WAPE and stock-weighted error. |
| Risks | Future reviews, realised future promotions or future stock status are leakage. Demand is censored by stock availability, which is not supplied. |

### ML Question 4: Are there useful customer segments beyond the published loyalty tiers?

| Element | Response |
|---|---|
| EDA evidence | Figures 1 and 8 show uneven tier sizes and descriptive operational differences that may hide other customer structures. |
| Decision and unit | Design differentiated communication or service research; one customer. |
| Type and objective | Clustering; identify stable groups using pre-period value, order frequency, channel/device preference and broad household/account attributes. |
| Validation and metric | Fit on an earlier cohort and assess stability on a later cohort; silhouette score plus cluster-size and temporal-stability checks. |
| Risks | Sensitive or proxy attributes may create unfair segments. Results require human interpretation, minimum cluster sizes and monitoring rather than automatic service denial. |

### ML Question 5: Can late delivery be predicted at dispatch time?

| Element | Response |
|---|---|
| EDA evidence | Figures 7-8 show overall delay prevalence and variation across carrier/service and customer groups. |
| Decision and unit | Trigger proactive intervention or communication; one dispatched order/delivery. |
| Type and target | Binary classification; `delay_days > 0`. |
| Decision-time predictors | Carrier, service level, promised days, shipping distance, signature flag, expedited flag, warehouse, order total and season, restricted to values known at dispatch. |
| Validation and metric | Time-based split; PR-AUC, delayed-delivery recall and calibration, with a simple baseline. |
| Risks | Delivered date, final status, delay reason and tracking outcomes are leakage. Loyalty tier should be excluded from primary decisions or separately fairness-audited; carrier/route drift is likely. |

## 5. Limitations

- The analysis covers one historical year, so monthly movement cannot establish recurring seasonality or future behaviour.
- All relationships are observational. Coupon use, carrier, service level, customer tier and product category were not randomly assigned.
- Reviews represent only customers who submitted reviews; product availability, exposure, stock-outs, marketing campaigns and acquisition costs are not observed.
- Category and subgroup summaries may hide within-group variation. Small Platinum and carrier-service subgroups create greater uncertainty than larger groups.
- Join multiplication was explicitly checked for Figures 6 and 8, but causal or deployment decisions would still require fresh-period validation and monitoring.

## 6. Conclusion

Group029's standardised relational outputs support a coherent view of customers, products, transactions, reviews and delivery operations. The strongest descriptive signals are the large category price differences, positive-skewed review ratings, monthly variation in order activity and revenue, and non-trivial delivery-delay differences across operational segments. These results identify useful follow-up questions, but they do not establish causation. Future work should prioritise time-aware validation, leakage control, join-grain discipline and fairness checks before any model or operational change is deployed.

## References

1. Monash University. (2026). *FIT5196 S2 2026 Assessment 1 Specification*.
2. Monash University. (2026). *FIT5196 S2 2026 Assessment 1 Marking Rubric*.
3. The pandas development team. (2026). *pandas documentation*.
4. Matplotlib Development Team. (2026). *Matplotlib documentation*.